# Odometria visual MONOCULAR e ESTÉREO — C++ e Python

Este notebook preserva a célula existente do Colab original e acrescenta uma demonstração reproduzível. Execute **Ambiente de execução → Executar tudo** em um runtime CPU novo. Os dados públicos são obtidos da fonte oficial; nenhum arquivo desta máquina é necessário.

O modo **mono** usa apenas a câmera esquerda: matriz essencial com RANSAC, cheiralidade e paralaxe compensada por rotação para inicializar um mapa; depois associa landmarks 3D e usa PnP robusto. A escala absoluta é desconhecida. A primeira baseline define apenas a unidade arbitrária do mapa; **não** há passos unitários a cada quadro.

O modo **stereo** usa pares IR ou grayscale esquerda/direita realmente sincronizados e calibração de ambas as câmeras. Triangulação estéreo fornece profundidade na unidade do baseline; PnP estima a pose seguinte. A escala é métrica apenas quando o baseline tem unidade física conhecida.

Referenciais ópticos OpenCV: x à direita, y para baixo, z para frente. `R_wc` transforma câmera→mundo do segmento e `C_w` é o centro da câmera. Para `X_atual = R_rel X_anterior + t_rel`:

```
R_wc_new = R_wc @ R_rel.T
C_w_new = C_w - escala * R_wc_new @ t_rel
```

O PnP com landmarks em coordenadas mundiais retorna mundo→câmera: `R_wc=R_cw.T`, `C_w=-R_cw.T@t_cw`. Perdas sem recuperação do mapa abrem outro segmento, sem continuidade ou escala inventada.


In [ ]:
import os, sys, json, shutil, subprocess, csv
from pathlib import Path
from IPython.display import display, Image, Markdown

RUNTIME_ROOT = Path(os.environ.get("VO_NOTEBOOK_ROOT", "/content/validated_vo" if Path("/content").exists() else str(Path.cwd()/"validated_vo_runtime"))).resolve()
RUNTIME_ROOT.mkdir(parents=True, exist_ok=True)
REPO = RUNTIME_ROOT/"Visao_Computacional"
REVISION = "a7ee87efb2fd6150505ba2b8a9fd1d8b86fd713f"
BRANCH = "feat/validated-mono-stereo-odometry"

def run(command, cwd=None, env=None):
    print("+", " ".join(map(str, command)), flush=True)
    subprocess.run(list(map(str, command)), cwd=cwd, env=env, check=True)

# Sparse clone: only this project, with an immutable implementation revision.
if not (REPO/".git").exists():
    run(["git", "clone", "--filter=blob:none", "--no-checkout", "--branch", BRANCH,
         "--single-branch", "https://github.com/wagnerdgarcia/Visao_Computacional.git", REPO])
    run(["git", "sparse-checkout", "init", "--cone"], cwd=REPO)
    run(["git", "sparse-checkout", "set", "Project/vision_odometry"], cwd=REPO)
run(["git", "fetch", "origin", BRANCH], cwd=REPO)
run(["git", "checkout", "--detach", REVISION], cwd=REPO)
PROJECT = REPO/"Project/vision_odometry"
print("Revisão fixa:", subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip())


## Dependências isoladas e build sem janelas

A execução usa um ambiente virtual próprio, evitando incompatibilidade entre NumPy e OpenCV já importados pelo kernel do Colab. Os programas C++ não usam HighGUI, viz ou Eigen; apenas os módulos necessários do OpenCV. A instalação apt é executada somente se as ferramentas de build estiverem ausentes.


In [ ]:
opencv_ok = shutil.which("pkg-config") and subprocess.run(["pkg-config", "--exists", "opencv4"]).returncode == 0
if not (opencv_ok and shutil.which("cmake") and shutil.which("g++")):
    prefix = [] if os.geteuid() == 0 else ["sudo"]
    run(prefix+["apt-get", "update", "-qq"])
    run(prefix+["apt-get", "install", "-y", "--no-install-recommends", "cmake", "g++", "pkg-config", "libopencv-dev", "python3-venv"])
ENV = RUNTIME_ROOT/".venv"
if not (ENV/"bin/python").exists():
    run([sys.executable, "-m", "venv", ENV])
PYTHON = ENV/"bin/python"
run([PYTHON, "-m", "pip", "install", "--quiet", "-r", PROJECT/"requirements.txt"])
run(["cmake", "-S", PROJECT, "-B", PROJECT/"build", "-DCMAKE_BUILD_TYPE=Release"])
run(["cmake", "--build", PROJECT/"build", "-j2"])
run([PYTHON, "-c", "import cv2,numpy; print('Python OpenCV',cv2.__version__,'NumPy',numpy.__version__)"])
run(["pkg-config", "--modversion", "opencv4"])


## Testes geométricos e de entradas

Verificamos composição com câmera rotacionada, escala e profundidade em cena 3D sintética, máscaras de outliers, reprojeção e retificação, repouso/rotação pura, falta de textura/descritores, resolução, pares ausentes/dessincronizados, perdas, reinicialização e ausência de atualização indevida. Os testes de CLI executam os dois programas. Teste sintético comprova a convenção implementada, não qualidade da calibração física.


In [ ]:
run(["ctest", "--test-dir", PROJECT/"build", "--output-on-failure"])
test_env = os.environ.copy()
test_env["PYTEST_DISABLE_PLUGIN_AUTOLOAD"] = "1"
run([PYTHON, "-m", "pytest", "tests", "-q"], cwd=PROJECT, env=test_env)


## Sequência pública pequena e calibrada

Usamos os mesmos 60 pares da sequência KITTI odometry **00**, quadros 0–59, nos dois modos. Mono recebe apenas a esquerda. O downloader lê partes dos ZIPs por HTTP Range, confere CRC e registra SHA-256; não baixa o arquivo completo de 23 GB.

[Dados oficiais](https://www.cvlibs.net/datasets/kitti/eval_odometry.php) · [bucket oficial](https://registry.opendata.aws/kitti/). Licença **CC BY-NC-SA 3.0**, para uso acadêmico não comercial; atribuição: Geiger, Lenz e Urtasun, CVPR 2012. Os dados não são incluídos no Git. Os números deste recorte curto não são um benchmark KITTI completo.


In [ ]:
# A second Run all may reuse only a complete cache with verified image hashes.
import hashlib
data_dir = PROJECT/"data/kitti00"
cache_ok = False
if (data_dir/"provenance.json").is_file():
    saved = json.loads((data_dir/"provenance.json").read_text())
    hashes = saved.get("hashes", [])
    required = ["calibration.json", "frames.csv", "reference.csv", "calib.txt", "times.txt", "poses.txt"]
    cache_ok = saved.get("sequence") == "00" and saved.get("start") == 0 and saved.get("frames") == 60 and len(hashes) == 120 and all((data_dir/f).is_file() for f in required)
    if cache_ok:
        for item in hashes:
            file = (data_dir/item["file"]).resolve()
            if data_dir.resolve() not in file.parents or not file.is_file() or hashlib.sha256(file.read_bytes()).hexdigest() != item["sha256"]:
                cache_ok = False
                break
if not cache_ok:
    run([PYTHON, "scripts/download_kitti.py", "--frames", "60", "--output", "data/kitti00"], cwd=PROJECT)
else:
    print("Cache oficial completo: 120 hashes de imagens verificados.")
provenance = json.loads((PROJECT/"data/kitti00/provenance.json").read_text())
print({k:v for k,v in provenance.items() if k != "hashes"})
print((PROJECT/"data/kitti00/calibration.json").read_text())


## Executar mono e stereo em C++ e Python

Cada execução escreve um CSV com índice, timestamp, segmento, estado, validade da pose, posição, orientação câmera→mundo, unidade, inliers, proporção, reprojeção, paralaxe, cheiralidade, erro epipolar e tamanho do mapa.

O estimador recebe **somente** imagens e calibração. A referência entra exclusivamente na avaliação posterior. O monocular é alinhado por Sim(3), uma escala global por segmento independente; o estéreo por SE(3), com escala fixada em 1. Nunca se ajusta escala estéreo para esconder erro de baseline.


In [ ]:
run([PYTHON, "scripts/run_real_validation.py", "--datasets", "kitti00", "--jobs", "2"], cwd=PROJECT)
for language in ["python", "cpp"]:
    for mode in ["mono", "stereo"]:
        stem = PROJECT/f"results/kitti00_{language}_{mode}"
        result = json.loads(stem.with_name(stem.name+"_evaluation").with_suffix(".json").read_text())
        print(language, mode, "aceitos:", result["accepted_count"], "/", result["frame_count"])
        print(json.dumps(result["segments"], indent=2))
        display(Image(filename=str(stem.with_suffix(".png"))))


## Comparação e interpretação

Cada curva é desenhada por segmento, sem unir mapas reinicializados. RMSE/ATE de posição e RPE de 1 s são calculados nas poses aceitas e pareadas, sem extrapolação da referência ou preenchimento de lacunas maiores que 0,35 s. Contagem e taxa de aceitação devem ser consideradas junto ao erro; excluir poses rejeitadas não demonstra continuidade.

O alinhamento Sim(3) monocular torna sua RMSE diretamente incomparável a uma RMSE estéreo sem escala como se fossem a mesma condição experimental. A escala aplicada é informada no JSON. As duas implementações podem divergir por versões de OpenCV/SIFT e amostragem RANSAC; não prometemos igualdade bit a bit.

Limitações: front-end SIFT esparso, cena predominantemente rígida, sem bundle adjustment, loop closure, IMU, relocalização global ou compensação de rolling shutter. Inicialização monocular recusa cena planar, baixa paralaxe e rotação pura; após mapa válido, PnP pode estimar rotação. Critérios conservadores podem causar perdas. Calibração e sincronização são necessárias; timestamps próximos não provam sincronização física perfeita.


## Dados da Arena do usuário (opcional)

Foram gerados três recortes reais de 20 s, 199 pares IR sincronizados cada, contendo imagens, IMUs, CameraInfo/TF registrados e trajetória FAST-LIO com timestamps originais. Esses dados não foram publicados no GitHub. Para usá-los, envie ao runtime o pacote `arena_vo_dados.zip` entregue com o projeto, ou coloque-o no Drive e defina seu caminho abaixo. O restante do notebook já executa integralmente sem esse arquivo.

A referência Arena é **FAST-LIO, não ground truth**. Aplicamos a cadeia TF registrada da IMU Livox à câmera para comparar o mesmo ponto, incluindo braço de alavanca. As extrínsecas de montagem nominais não foram aferidas fisicamente neste trabalho; os resultados são discrepância entre métodos. A IMU exportada está disponível para inspeção/extensão VIO e não alimenta a odometria visual.

Para extrair novamente de MCAP: instale `requirements-bags.txt` e execute `scripts/export_arena.py --help`. O README documenta tópicos, CameraInfo.P dos fluxos retificados e limitações das calibrações.


In [ ]:
# Opcional: no Colab, envie o ZIP pelo painel Arquivos, ou monte o Drive.
# from google.colab import drive
# drive.mount('/content/drive')
ARENA_ZIP = None  # Ex.: '/content/arena_vo_dados.zip'
if ARENA_ZIP:
    import zipfile
    archive = Path(ARENA_ZIP).resolve()
    destination = PROJECT/"data"
    destination.mkdir(exist_ok=True)
    with zipfile.ZipFile(archive) as z:
        # Validate traversal and links before extracting user-provided archives.
        for entry in z.infolist():
            target = (destination/entry.filename).resolve()
            if destination.resolve() not in target.parents or ((entry.external_attr >> 16) & 0o170000) == 0o120000:
                raise ValueError('Unsafe ZIP member')
        z.extractall(destination)
    run([PYTHON, "scripts/run_real_validation.py", "--datasets", "arena1", "arena2", "arena3", "--jobs", "2"], cwd=PROJECT)
    for dataset in ["arena1", "arena2", "arena3"]:
        for mode in ["mono", "stereo"]:
            stem = PROJECT/f"results/{dataset}_python_{mode}"
            print(dataset, mode, json.loads(stem.with_name(stem.name+"_evaluation").with_suffix(".json").read_text())["meaning"])
            display(Image(filename=str(stem.with_suffix(".png"))))
else:
    print('Seção Arena opcional: nenhum ZIP fornecido. Demonstração pública completa acima.')


## Exportar resultados

Os CSVs ficam em `Project/vision_odometry/results/`. Os JSONs de execução registram comandos e versões são exibidas acima. Guarde os CSVs, configurações, SHA-256 dos dados e revisão do código para reproduzir a análise. A célula a seguir cria um ZIP com resultados desta sessão; caches de renderização não são incluídos.


In [ ]:
import zipfile
output_zip = RUNTIME_ROOT/"vo_resultados_colab.zip"
with zipfile.ZipFile(output_zip, "w", zipfile.ZIP_DEFLATED) as z:
    for pattern in ["*.csv", "*.json", "*.png"]:
        for path in sorted((PROJECT/"results").glob(pattern)):
            z.write(path, "results/"+path.name)
print("Resultados:", output_zip)
# Para baixar no Colab:
# from google.colab import files
# files.download(str(output_zip))
